<a href="https://colab.research.google.com/github/internshipinabook/nlp-internship-in-a-book/blob/main/notebooks/week12_capstone_STARTER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ⏸️ **Pause and Predict**
> Before writing any capstone code: write a one-paragraph description of what you are building. What problem? What data? What model? What does Week 0 you not know that Week 12 you knows?

In [ ]:
# ── Colab setup (skip if running locally) ──────────────────────
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/nlp-internship-in-a-book.git book
    %cd book
    !pip install -r requirements.txt -q
    print('Colab setup complete ✅')
else:
    print('Running locally ✅')


# Week 12 — The Capstone (STARTER)
### The NLP Internship | LinguaAI Labs

**Dataset:** AfriSenti — real Nigerian Pidgin and Yorùbá tweets.
Peer-reviewed: Muhammad et al. (2023) EMNLP. CC BY 4.0.

**Question:** How does sentiment differ across Nigerian language groups, and is a zero-shot English model equally fair to both?

**Write your problem statement before loading data.**

## Problem Statement

**What are we predicting or discovering?** [WRITE HERE]
**Why does it matter?** [WRITE HERE]
**What does success look like?** [WRITE HERE — concrete, measurable]
**What could go wrong?**
1. [WRITE HERE]
2. [WRITE HERE]

In [ ]:
import sys, subprocess, os, re
for pkg in ["datasets","transformers"]:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install",pkg,"-q"])
from datasets import load_dataset
from transformers import pipeline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics import f1_score
import warnings; warnings.filterwarnings("ignore")
np.random.seed(42); os.makedirs("outputs",exist_ok=True)

print("Loading AfriSenti PCM (Nigerian Pidgin)...")
afri_pcm = load_dataset("shmuhammad/AfriSenti-twitter-sentiment", "pcm")
df_pcm = afri_pcm["train"].to_pandas().rename(columns={"tweet":"text"})
df_pcm["language"] = "Pidgin"

print("Loading AfriSenti YO (Yorùbá)...")
afri_yo = load_dataset("shmuhammad/AfriSenti-twitter-sentiment", "yo")
df_yo = afri_yo["train"].to_pandas().rename(columns={"tweet":"text"})
df_yo["language"] = "Yoruba"

df = pd.concat([df_pcm, df_yo], ignore_index=True)
LABEL_MAP = {0:"negative", 1:"neutral", 2:"positive"}
df["label_name"] = df["label"].map(LABEL_MAP)
print(f"AfriSenti: {len(df):,} tweets | {df['language'].value_counts().to_dict()}")
print("📚 Muhammad et al. (2023) AfriSenti EMNLP — CC BY 4.0 ✅")

## Monday — First Look

> ⏸️ **Predict:** which language will have more negative sentiment — Pidgin or Yorùbá? Why?

In [ ]:
print("=== AfriSenti Overview ===\n")
print(df.groupby(["language","label_name"]).size().unstack(fill_value=0))
df["wc"]=df["text"].str.split().str.len()
print(f"\nMean word count: {df.groupby('language')['wc'].mean().round(1).to_dict()}")
print(f"\nSample Pidgin: {df[df['language']=='Pidgin']['text'].iloc[0]}")
print(f"Sample Yorùbá: {df[df['language']=='Yoruba']['text'].iloc[0]}")

## Tuesday — Cleaning and Topic Discovery

In [ ]:
def clean_tweet(text):
    if not isinstance(text, str): return ""
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    text = re.sub(r"[^\w\s.,!?]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

df["text_clean"] = df["text"].apply(clean_tweet)

df_pcm_clean = df[df["language"]=="Pidgin"]["text_clean"].dropna()
lda_vec = CountVectorizer(max_features=1000, stop_words="english", min_df=3, max_df=0.90)
dtm = lda_vec.fit_transform(df_pcm_clean)
vocab = lda_vec.get_feature_names_out()
lda = LatentDirichletAllocation(n_components=5, random_state=42, max_iter=20, learning_method="online")
lda.fit(dtm)
print("Topics in Pidgin tweets:\n")
for i, topic in enumerate(lda.components_):
    top = [vocab[j] for j in topic.argsort()[::-1][:8]]
    print(f"  Topic {i+1}: {' | '.join(top)}")
# YOUR CODE HERE — name each topic in a markdown cell below

## Wednesday — Zero-Shot Sentiment

> 🧠 **Predict:** will the zero-shot model (English BART) be more or less confident on Pidgin vs Yorùbá? Why?

In [ ]:
print("Loading zero-shot pipeline (~1GB first run)...")
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
sample = df.sample(min(100, len(df)), random_state=42).copy()
print(f"Classifying {len(sample)} tweets...")
labels_out, confs = [], []
for text in sample["text_clean"]:
    r = zs(text[:512], ["positive","negative","neutral"])
    labels_out.append(r["labels"][0]); confs.append(r["scores"][0])
sample["zs_label"]=labels_out; sample["zs_conf"]=confs
print(f"\nZero-shot distribution: {sample['zs_label'].value_counts().to_dict()}")
print(f"\nMean confidence by language:")
print(sample.groupby("language")["zs_conf"].mean().round(3))

## Thursday — Language-Group Fairness Audit

In [ ]:
for lang in ["Pidgin","Yoruba"]:
    mask=sample["language"]==lang; n=mask.sum()
    if n<5: continue
    mc=sample.loc[mask,"zs_conf"].mean()
    sd=sample.loc[mask,"zs_label"].value_counts(normalize=True)
    print(f"  {lang}: n={n} | conf={mc:.3f} | {dict(sd.round(2))}")

# YOUR CODE HERE — compute confidence gap and interpret
pidgin_conf = None  # YOUR CODE HERE
yoruba_conf = None  # YOUR CODE HERE
gap = None          # YOUR CODE HERE
print(f"\nConfidence gap (Pidgin vs Yorùbá): {gap:.3f}")
if gap and abs(gap) > 0.10:
    print("⚠️  Gap > 10% — document in fairness brief")
    print("Root cause: BART pre-trained on English — both languages equally foreign to it")

## Friday — Brief and Reflection

## AfriSenti Sentiment Analysis — Fairness Brief

**Pidgin sentiment:** [fill in] | **Yorùbá sentiment:** [fill in]
**Zero-shot confidence gap:** [fill in]
**Model limitation:** BART pre-trained on English — neither language in training data.
**Recommendation:** Use AfriSenti labels (real human annotations) — not zero-shot predictions — for production decisions. Cite Muhammad et al. (2023) in any publication.

In [ ]:
LETTER="""Portfolio Letter — Week 12 Capstone
Dataset: AfriSenti (Muhammad et al. 2023 EMNLP) — Nigerian Pidgin + Yorùbá sentiment
What: Topic discovery + zero-shot sentiment + language-group fairness audit
Finding I did not expect: [YOUR ANSWER]
What I would do differently: [YOUR ANSWER]
The fairness gap is documented. Not hidden. That is the most honest part.
--- [Your Name], [Date]"""
with open("outputs/portfolio_letter.md","w") as f: f.write(LETTER)
print("Portfolio letter saved ✅")

## ⚠️ Common Mistakes

| Mistake | What goes wrong | Fix |
|---------|-----------------|-----|
| Submitting a notebook full of exploratory dead ends | Your capstone is a professional document. All exploratory cells belong in a scratch notebook. Only the cells that tell the story should be in the final submission. | Final notebook: every cell earns its place |
| Skipping the retrospective | The retrospective is how interviewers know you can reflect and grow. 'I built X' is less impressive than 'I built X, it failed because Y, and I learned Z'. | Write the retrospective even if everything went well — especially then |

## 🏆 Challenge Task

> 🎯 **Core Path**
> Assemble your complete portfolio: clean notebooks, model card, deployed app, and a polished README. Push with a v1.0 release tag.

> 🔬 **Advanced Path**
> Write a one-page retrospective. What would you do differently in Weeks 2, 5, and 8? What is the single most important thing you learned about NLP that surprised you?

## ✅ What You Can Do After This Week

- Load and analyse a real peer-reviewed African language NLP dataset
- Apply topic modelling to non-English social media text
- Run zero-shot sentiment analysis and interpret confidence scores
- Conduct a language-group fairness audit with quantitative findings

---
```
╔═══════════════════════════════════════════╗
║        CERTIFICATE OF COMPLETION         ║
║     THE NLP INTERNSHIP — BOOK 2 OF 7    ║
║  _____________________________           ║
║  LinguaAI Labs · Lagos · 2024            ║
║  Date: ______________________            ║
╚═══════════════════════════════════════════╝
```
---
## ✅ Book 2 Complete
*Add `week12_capstone_STARTER.ipynb` to your portfolio.*

*The NLP Internship · LinguaAI Labs · github.com/InternshipInABook*

## ✅ By completing Week 12, you can now:

- Assemble a complete end-to-end NLP portfolio from 12 weeks of work
- Write a retrospective identifying what you would do differently
- Produce a model card and fairness brief suitable for a real deployment team
- Prepare your GitHub repository for public visibility — recruiter-ready

📤 **GitHub:** Final push: all notebooks, model card, fairness brief, app.py, README. Tag v1.0. Commit: "Week 12: Capstone complete"


---

## 📚 Get the Full Book

This notebook is part of **The NLP Internship** — Book 2 of the InternshipInABook™ Series.

The full book includes:
- Complete week-by-week narrative and explanations
- All STOP AND TRACE code walkthroughs
- Fairness briefs, model cards, and deployment guides
- Certificate of Completion

👉 **[Get the book on Selar →](https://selar.com/8440iqfr61)**

---
*InternshipInABook™ Series · © Sakinat Folorunso 2026 · [github.com/internshipinabook](https://github.com/internshipinabook)*
